In [1]:
import pandas as pd
df=pd.read_csv('supermarket_sales.csv')

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   str    
 1   Branch                   1000 non-null   str    
 2   City                     1000 non-null   str    
 3   Customer type            1000 non-null   str    
 4   Gender                   1000 non-null   str    
 5   Product line             1000 non-null   str    
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Total                    1000 non-null   float64
 10  Date                     1000 non-null   str    
 11  Time                     1000 non-null   str    
 12  Payment                  996 non-null    str    
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  1000 non-nu

In [3]:
df.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


In [4]:
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(' ','_')

In [5]:
df.columns

Index(['invoice_id', 'branch', 'city', 'customer_type', 'gender',
       'product_line', 'unit_price', 'quantity', 'tax_5%', 'total', 'date',
       'time', 'payment', 'cogs', 'gross_margin_percentage', 'gross_income',
       'rating'],
      dtype='str')

In [7]:
df.isnull().sum()

invoice_id                 0
branch                     0
city                       0
customer_type              0
gender                     0
product_line               0
unit_price                 0
quantity                   0
tax_5%                     0
total                      0
date                       0
time                       0
payment                    4
cogs                       0
gross_margin_percentage    0
gross_income               0
rating                     7
dtype: int64

In [10]:
df=df.rename(columns={'rating':'customer_rating'})
df.columns

Index(['invoice_id', 'branch', 'city', 'customer_type', 'gender',
       'product_line', 'unit_price', 'quantity', 'tax_5%', 'total', 'date',
       'time', 'payment', 'cogs', 'gross_margin_percentage', 'gross_income',
       'customer_rating'],
      dtype='str')

In [11]:
df['payment']=df.groupby('customer_type')['payment'].transform(lambda x:x.fillna(x.mode()[0]))
df['customer_rating']=df.groupby('product_line')['customer_rating'].transform(lambda x:x.fillna(x.mean()))
df.isnull().sum()

invoice_id                 0
branch                     0
city                       0
customer_type              0
gender                     0
product_line               0
unit_price                 0
quantity                   0
tax_5%                     0
total                      0
date                       0
time                       0
payment                    0
cogs                       0
gross_margin_percentage    0
gross_income               0
customer_rating            0
dtype: int64

In [12]:
labels=['Below rated', 'Average','Top Rated']
df['rating_category']=pd.qcut(df['customer_rating'],q=3,labels =labels)
df[['customer_rating', 'rating_category']].head()

,customer_rating,rating_category
0,9.1,Top Rated
1,9.6,Top Rated
2,7.4,Average
3,8.4,Top Rated
4,5.3,Below rated


In [14]:

frequency_mapping ={
    'Cash':'Offline',
    'Ewallet':'Digital',
    'Credit card':'Card'
}
df['payment_type']=df['payment'].map(frequency_mapping)
df[['payment','payment_type']].head()

,payment,payment_type
0,Ewallet,Digital
1,Cash,Offline
2,Credit card,Card
3,Ewallet,Digital
4,Ewallet,Digital


In [15]:
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Length: 1000, dtype: bool

In [16]:
import pymysql
import sqlalchemy
print("Installed successfully")

Installed successfully


In [17]:
from sqlalchemy import create_engine
username="root"
password="Vijay1234!"
host="localhost"
port="3306"
database="superdata"

engine= create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
table_name="mytable"
df.to_sql(table_name, engine, if_exists="replace", index= False)

pd.read_sql("select*from mytable limit 5;",engine)

,invoice_id,branch,city,customer_type,gender,product_line,unit_price,quantity,tax_5%,total,date,time,payment,cogs,gross_margin_percentage,gross_income,customer_rating,rating_category,payment_type
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1,Top Rated,Digital
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6,Top Rated,Offline
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4,Average,Card
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4,Top Rated,Digital
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3,Below rated,Digital


In [1]:
df.info()

NameError: name 'df' is not defined